# Thực nghiệm Pre-trained Language Models: ELMo, BERT và GPT

**Chạy trên Google Colab:** mở file `.ipynb` bằng Google Colab, chọn Runtime → Change runtime type → (GPU tùy chọn), sau đó chạy lần lượt các cell.

Notebook minh họa ba hướng mô hình:
- **ELMo:** trích xuất contextual embeddings từ mô hình BiLSTM hai chiều (TensorFlow Hub).
- **BERT:** phân tích cảm xúc bằng một mô hình BERT fine-tuned.
- **GPT:** sinh văn bản tiếp theo bằng GPT-2.

> Lưu ý: bài khảo sát nguồn được xuất bản năm 2022. Notebook dùng các checkpoint công khai hiện có để minh họa cách sử dụng; kết quả demo không phải phép so sánh benchmark công bằng giữa ba họ mô hình.

## 0. Cài đặt thư viện

In [1]:
!pip -q install transformers==4.44.2 tensorflow-hub==0.16.1

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  error: subprocess-exited-with-error
  
  × Building wheel for tokenizers (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> No available output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for tokenizers
error: failed-wheel-build-for-install

× Failed to build installable wheels for some pyproject.toml based projects
╰─> tokenizers


In [2]:
import torch, numpy as np
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

PyTorch: 2.14.1+cpu
CUDA available: False


## 1. ELMo – contextual word embeddings

ELMo dùng các trạng thái bên trong của một bidirectional language model dựa trên LSTM. Embedding của từ phụ thuộc vào câu chứa từ đó.

Cell dưới đây thử tải ELMo từ TensorFlow Hub. Đây là mô hình đời cũ, nên khả năng tương thích phụ thuộc phiên bản TensorFlow/Colab. Nếu cell báo lỗi, ghi nhận lỗi và tiếp tục sang BERT/GPT; hai phần sau không phụ thuộc ELMo.

In [3]:
# ELMo demo (TensorFlow Hub)
elmo_ok = False
try:
    import tensorflow as tf
    import tensorflow_hub as hub

    print("TensorFlow:", tf.__version__)
    elmo = hub.load("https://tfhub.dev/google/elmo/3")
    elmo_fn = elmo.signatures["default"]

    sentences = [
        ["I", "went", "to", "the", "bank", "to", "deposit", "money", "."],
        ["We", "had", "a", "picnic", "on", "the", "river", "bank", "."]
    ]
    lengths = tf.constant([len(s) for s in sentences], dtype=tf.int32)
    max_len = max(map(len, sentences))
    padded = [s + [""] * (max_len-len(s)) for s in sentences]
    tokens = tf.constant(padded)

    result = elmo_fn(tokens=tokens, sequence_len=lengths)
    print("Output keys:", list(result.keys()))
    embeddings = result["elmo"].numpy()
    print("ELMo embedding shape:", embeddings.shape)
    print("Embedding của 'bank' trong câu 1:", embeddings[0, 4, :8])
    print("Embedding của 'bank' trong câu 2:", embeddings[1, 6, :8])
    elmo_ok = True
except Exception as e:
    print("Không tải/chạy được ELMo trong runtime hiện tại.")
    print("Chi tiết lỗi:", repr(e))
    print("Có thể tiếp tục thực nghiệm BERT và GPT ở các phần dưới.")

/usr/local/lib/python3.13/dist-packages/tensorflow_hub/__init__.py:61: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version


TensorFlow: 2.21.0
Không tải/chạy được ELMo trong runtime hiện tại.
Chi tiết lỗi: TypeError("Binding inputs to tf.function failed due to `missing a required argument: 'text'`. Received args: () and kwargs: {'tokens': <tf.Tensor: shape=(2, 9), dtype=string, numpy=\narray([[b'I', b'went', b'to', b'the', b'bank', b'to', b'deposit',\n        b'money', b'.'],\n       [b'We', b'had', b'a', b'picnic', b'on', b'the', b'river', b'bank',\n        b'.']], dtype=object)>, 'sequence_len': <tf.Tensor: shape=(2,), dtype=int32, numpy=array([9, 9], dtype=int32)>} for signature: (text: TensorSpec(shape=(None,), dtype=tf.string, name='text')) -> Dict[['lstm_outputs2', TensorSpec(shape=(None, None, 1024), dtype=tf.float32, name=None)], ['default', TensorSpec(shape=(None, 1024), dtype=tf.float32, name=None)], ['lstm_outputs1', TensorSpec(shape=(None, None, 1024), dtype=tf.float32, name=None)], ['elmo', TensorSpec(shape=(None, None, 1024), dtype=tf.float32, name=None)], ['word_emb', TensorSpec(shape=(None, 

### Nhận xét ELMo

- Nếu tải thành công, `embeddings` có dạng `(số_câu, độ_dài_chuỗi, chiều_embedding)`.
- So sánh vector của từ *bank* trong hai câu: một câu nói về ngân hàng, câu còn lại nói về bờ sông.
- Nếu hai vector khác nhau, đó là minh họa cho biểu diễn từ phụ thuộc ngữ cảnh.
- Nếu không tải được, nguyên nhân thường nằm ở tương thích thư viện/mô hình Hub cũ hoặc kết nối tải checkpoint; không ảnh hưởng đến hai phần tiếp theo.

## 2. BERT Family – phân tích cảm xúc

Dùng checkpoint `distilbert-base-uncased-finetuned-sst-2-english`, một mô hình thuộc dòng BERT/DistilBERT đã fine-tune cho phân loại cảm xúc nhị phân. Mô hình nhận câu đầu vào và trả về nhãn cùng điểm số.

In [4]:
from transformers import pipeline
import torch

device = 0 if torch.cuda.is_available() else -1
sentiment = pipeline(
    "sentiment-analysis",
    model="distilbert/distilbert-base-uncased-finetuned-sst-2-english",
    device=device
)

texts = [
    "This movie is wonderful. I really enjoyed it.",
    "The service was terrible and I am disappointed.",
    "The food was okay, but nothing special."
]
bert_results = sentiment(texts)
for text, result in zip(texts, bert_results):
    print("Câu:", text)
    print("Kết quả:", result)
    print("-" * 60)

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

Câu: This movie is wonderful. I really enjoyed it.
Kết quả: {'label': 'POSITIVE', 'score': 0.9998819828033447}
------------------------------------------------------------
Câu: The service was terrible and I am disappointed.
Kết quả: {'label': 'NEGATIVE', 'score': 0.9996482133865356}
------------------------------------------------------------
Câu: The food was okay, but nothing special.
Kết quả: {'label': 'NEGATIVE', 'score': 0.9937840700149536}
------------------------------------------------------------


### Thử thay đổi câu đầu vào

Sửa nội dung `my_texts` để thử các câu tích cực, tiêu cực hoặc trung tính. Lưu ý checkpoint này được huấn luyện cho tiếng Anh; không nên dùng kết quả của nó để kết luận về câu tiếng Việt.

In [5]:
my_texts = [
    "I love learning natural language processing.",
    "This assignment is confusing and difficult.",
    "The weather is cloudy today."
]
for text, result in zip(my_texts, sentiment(my_texts)):
    print(f"{text}\n=> {result}\n")

I love learning natural language processing.
=> {'label': 'POSITIVE', 'score': 0.9997851252555847}

This assignment is confusing and difficult.
=> {'label': 'NEGATIVE', 'score': 0.9993966817855835}

The weather is cloudy today.
=> {'label': 'NEGATIVE', 'score': 0.9806528091430664}



## 3. GPT Family – sinh văn bản

Dùng GPT-2 để minh họa mô hình autoregressive: mô hình dự đoán token tiếp theo dựa trên các token đã có. Thay đổi prompt để quan sát nội dung sinh ra.

In [6]:
generator = pipeline(
    "text-generation",
    model="gpt2",
    device=device
)

prompt = "Natural language processing is an important field because"
generated = generator(
    prompt,
    max_new_tokens=60,
    do_sample=True,
    temperature=0.8,
    top_p=0.9,
    num_return_sequences=1,
    pad_token_id=50256
)
print(generated[0]["generated_text"])

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Natural language processing is an important field because it facilitates rapid learning of new vocabulary concepts. As language learning improves, researchers are now developing new ways to understand new concepts and languages. The next step is to understand the language and its components as best as possible and to identify the best way to use language. This work is currently underway in the field


In [7]:
# Thử prompt khác
prompts = [
    "Artificial intelligence can help students by",
    "A pretrained language model learns from"
]
for prompt in prompts:
    result = generator(
        prompt, max_new_tokens=45, do_sample=True,
        temperature=0.8, top_p=0.9, pad_token_id=50256
    )[0]["generated_text"]
    print("PROMPT:", prompt)
    print("GENERATED:", result)
    print("-" * 80)

PROMPT: Artificial intelligence can help students by
GENERATED: Artificial intelligence can help students by introducing new skills, such as problem solving, to more people.

"It's a great way to learn and to be a good teacher," said Eric Heyer, an adjunct professor at the University of California, Santa
--------------------------------------------------------------------------------
PROMPT: A pretrained language model learns from
GENERATED: A pretrained language model learns from this. It learns from the context.

The training process is very easy, and the initial task is the same.

Training to learn from the context

You learn from a background knowledge. This may include
--------------------------------------------------------------------------------


## 4. Tổng hợp kết quả thực nghiệm

Điền nhận xét sau khi chạy notebook:

| Mô hình | Thực nghiệm | Điều cần quan sát |
|---|---|---|
| ELMo | Trích xuất embedding của từ *bank* trong hai câu | Vector phụ thuộc vào ngữ cảnh; ghi lại nếu môi trường không tải được mô hình |
| BERT Family | Phân loại cảm xúc | Nhãn POSITIVE/NEGATIVE và điểm số |
| GPT Family | Sinh tiếp văn bản từ prompt | Nội dung thay đổi theo prompt và tham số sinh |

